# E8^k HAWK-Style Signing Stage

This notebook combines the hash to shift stage and the bootstrap coset sampler. It computes a transcript dependent shift `t` in `E8^k / 2E8^k`, then samples from the coset `t + 2E8^k` blockwise using the `E8` sampler. HAWK style signing stage for experimentation, not implementation of the full HAWK scheme.

In [1]:
from sage.stats.distributions.discrete_gaussian_integer import DiscreteGaussianDistributionIntegerSampler as DGaussZ
from sage.all import vector, matrix, ZZ, QQ, codes
from mpmath import mp, exp, jtheta, sqrt, pi, log

import itertools
import random
import hashlib
import os

mp.dps = 80

In [2]:
C = codes.BinaryReedMullerCode(1, 3)
CODEWORD_TUPLES = [tuple(int(x) for x in cw) for cw in C]
CODEWORD_SET = set(CODEWORD_TUPLES)

# Construction A setup for E8 = RM(1,3) + 2 Z^8
I2 = [vector(ZZ, [2 if i == j else 0 for j in range(8)]) for i in range(8)]
G = C.generator_matrix()
Grows = [vector(ZZ, [int(x) for x in G.row(i)]) for i in range(G.nrows())]
E8_BASIS = matrix(ZZ, I2 + Grows).row_module().basis_matrix()


def e8_mod_2e8_representatives():
    reps = []
    for bits in itertools.product([0, 1], repeat=8):
        r = vector(ZZ, [0] * 8)
        for i, b in enumerate(bits):
            if b:
                r += E8_BASIS.row(i)
        reps.append(r)
    return reps


def direct_sum_e8_basis(k):
    if k <= 0:
        raise ValueError("k must be positive")
    dim = E8_BASIS.nrows()
    basis = matrix(ZZ, dim * k, dim * k)
    for block in range(k):
        offset = block * dim
        for i in range(dim):
            for j in range(dim):
                basis[offset + i, offset + j] = E8_BASIS[i, j]
    return basis


REPS = e8_mod_2e8_representatives()
print("single-block quotient size:", len(REPS))
print("E8 basis shape:", E8_BASIS.nrows(), "x", E8_BASIS.ncols())

single-block quotient size: 256
E8 basis shape: 8 x 8


In [3]:
def nome(s):
    return exp(-pi / (s**2))


def rho_2Z_shift(alpha, s):
    q = nome(s)
    z = -2j * pi * alpha / (s**2)
    value = (q ** (alpha**2)) * jtheta(3, z, q**4)
    return float(value.real)


def codeword_weights_shifted_E8(s, shift):
    shift_f = [float(a) for a in shift]
    log_weights = []
    for c in CODEWORD_TUPLES:
        lw = 0.0
        for i in range(8):
            lw += float(log(rho_2Z_shift(c[i] + shift_f[i], s)))
        log_weights.append(lw)
    m = max(log_weights)
    return [float(exp(w - m)) for w in log_weights]


def sample_coord_2Z_coset(alpha, s):
    sigma = float(s) / (2 * float(sqrt(2 * pi)))
    center = -QQ(alpha) / 2
    z = DGaussZ(sigma=sigma, c=center)()
    return ZZ(2 * z) + QQ(alpha)


def sample_shifted_E8(s, shift):
    if len(shift) != 8:
        raise ValueError("shift must have length 8")

    weights = codeword_weights_shifted_E8(s, shift)
    idx = random.choices(range(len(CODEWORD_TUPLES)), weights=weights, k=1)[0]
    c = CODEWORD_TUPLES[idx]
    x = [sample_coord_2Z_coset(c[i] + shift[i], s) for i in range(8)]
    return vector(QQ, x), c


def sample_coset_rep_plus_2E8(s, rep):
    shift = [QQ(rep[i]) / 2 for i in range(8)]
    y, _ = sample_shifted_E8(s / 2, shift)
    return vector(QQ, [2 * yi for yi in y])


def in_E8(v):
    if len(v) != 8:
        return False
    for x in v:
        if x not in ZZ:
            return False
    parity = tuple(int(ZZ(x) % 2) for x in v)
    return parity in CODEWORD_SET


def in_2E8(v):
    if len(v) != 8:
        return False
    if any(ZZ(x) % 2 != 0 for x in v):
        return False
    half = vector(ZZ, [ZZ(x) // 2 for x in v])
    return in_E8(half)


def in_rep_plus_2E8(v, rep):
    diff = vector(ZZ, [ZZ(v[i] - rep[i]) for i in range(8)])
    return in_2E8(diff)


def split_e8_blocks(v):
    if len(v) % 8 != 0:
        raise ValueError("vector length must be a multiple of 8")
    return [vector(v.base_ring(), list(v[8 * i:8 * (i + 1)])) for i in range(len(v) // 8)]


def sample_direct_sum_coset_rep_plus_2E8(s, t):
    blocks = split_e8_blocks(t)
    samples = [sample_coset_rep_plus_2E8(s, block) for block in blocks]
    out = []
    for sample in samples:
        out.extend(sample)
    return vector(QQ, out)


def in_direct_sum_rep_plus_2E8(v, t):
    v_blocks = split_e8_blocks(v)
    t_blocks = split_e8_blocks(t)
    return all(in_rep_plus_2E8(vb, tb) for vb, tb in zip(v_blocks, t_blocks))


def lattice_point_to_basis_coefficients(v, basis):
    v_q = vector(QQ, v)
    coeffs_q = basis.transpose().solve_right(v_q.column()).column(0)
    if any(c not in ZZ for c in coeffs_q):
        raise ValueError("lattice point does not have integral coordinates in the chosen basis")
    return vector(ZZ, [ZZ(c) for c in coeffs_q])

In [4]:
def _normalize_binary_message(msg_bits):
    if isinstance(msg_bits, str):
        if any(ch not in "01" for ch in msg_bits):
            raise ValueError("bitstring must only contain '0' and '1'")
        return msg_bits

    try:
        bits = []
        for b in msg_bits:
            if int(b) not in (0, 1):
                raise ValueError
            bits.append(str(int(b)))
        return "".join(bits)
    except Exception as exc:
        raise ValueError("msg_bits must be a bitstring or iterable of bits") from exc


def _pack_bitstring(bitstring):
    bitlen = len(bitstring)
    if bitlen == 0:
        return 0, b""
    pad = (-bitlen) % 8
    padded = bitstring + ("0" * pad)
    value = int(padded, 2)
    return bitlen, value.to_bytes(len(padded) // 8, "big")


def _normalize_salt(salt):
    if isinstance(salt, bytes):
        return salt
    if isinstance(salt, str):
        s = salt[2:] if salt.startswith("0x") else salt
        if len(s) % 2 != 0:
            s = "0" + s
        try:
            return bytes.fromhex(s)
        except ValueError as exc:
            raise ValueError("salt hex string is invalid") from exc
    raise ValueError("salt must be bytes or hex string")


def _normalize_pubkey(pubkey_bytes):
    if pubkey_bytes is None:
        return b""
    if isinstance(pubkey_bytes, bytes):
        return pubkey_bytes
    if isinstance(pubkey_bytes, str):
        s = pubkey_bytes[2:] if pubkey_bytes.startswith("0x") else pubkey_bytes
        if len(s) % 2 != 0:
            s = "0" + s
        try:
            return bytes.fromhex(s)
        except ValueError as exc:
            raise ValueError("pubkey hex string is invalid") from exc
    raise ValueError("pubkey_bytes must be None, bytes, or hex string")


def build_hash_transcript(msg_bits, salt, pubkey_bytes=None, domain_sep=b"E8-quotient-shift-v1"):
    bitstring = _normalize_binary_message(msg_bits)
    bitlen, packed = _pack_bitstring(bitstring)
    salt_b = _normalize_salt(salt)
    pub_b = _normalize_pubkey(pubkey_bytes)
    transcript = (
        domain_sep
        + len(pub_b).to_bytes(4, "big") + pub_b
        + len(salt_b).to_bytes(2, "big") + salt_b
        + bitlen.to_bytes(8, "big") + packed
    )
    return transcript, salt_b


def hash_to_basis_coefficients(msg_bits, salt, pubkey_bytes=None, k=1, coeff_bytes=4):
    if coeff_bytes <= 0:
        raise ValueError("coeff_bytes must be positive")
    basis = direct_sum_e8_basis(k)
    transcript, salt_b = build_hash_transcript(msg_bits, salt, pubkey_bytes=pubkey_bytes)
    dim = basis.nrows()
    stream = hashlib.shake_256(transcript).digest(dim * coeff_bytes)
    coeffs = []
    for i in range(dim):
        start = i * coeff_bytes
        coeffs.append(ZZ(int.from_bytes(stream[start:start + coeff_bytes], "big")))
    return vector(ZZ, coeffs), basis, salt_b


def reduce_coefficients_mod_2e8(coeffs, basis):
    coeff_vec = vector(ZZ, coeffs)
    quotient_bits = vector(ZZ, [ZZ(c % 2) for c in coeff_vec])
    rep = quotient_bits * basis
    lift_coeffs = vector(ZZ, [(coeff_vec[i] - quotient_bits[i]) // 2 for i in range(len(coeff_vec))])
    return quotient_bits, rep, lift_coeffs


def hash_message_to_shift(msg_bits, salt=None, pubkey_bytes=None, k=1, coeff_bytes=4):
    if salt is None:
        salt_b = os.urandom(16)
    else:
        salt_b = _normalize_salt(salt)

    coeffs, basis, salt_b = hash_to_basis_coefficients(
        msg_bits,
        salt_b,
        pubkey_bytes=pubkey_bytes,
        k=k,
        coeff_bytes=coeff_bytes,
    )
    x = coeffs * basis
    quotient_bits, t, lift_coeffs = reduce_coefficients_mod_2e8(coeffs, basis)
    return {
        "salt": salt_b,
        "k": k,
        "basis": basis,
        "coeffs": coeffs,
        "x": x,
        "quotient_bits": quotient_bits,
        "lift_coeffs": lift_coeffs,
        "t": vector(ZZ, t),
    }

In [5]:
def hawk_style_e8_sign(msg_bits, pubkey_bytes, s, salt=None, k=1, coeff_bytes=4):
    shift_data = hash_message_to_shift(
        msg_bits,
        salt=salt,
        pubkey_bytes=pubkey_bytes,
        k=k,
        coeff_bytes=coeff_bytes,
    )
    t = shift_data["t"]
    sample = sample_direct_sum_coset_rep_plus_2E8(s, t)
    if not in_direct_sum_rep_plus_2E8(sample, t):
        raise RuntimeError("sampler output does not lie in the requested coset")
    w = lattice_point_to_basis_coefficients(sample, shift_data["basis"])
    h = shift_data["quotient_bits"]
    if any((w[i] - h[i]) % 2 != 0 for i in range(len(h))):
        raise RuntimeError("sample coordinates are not congruent to h modulo 2")
    s_vec = vector(ZZ, [(h[i] - w[i]) // 2 for i in range(len(h))])

    return {
        "message_bits": _normalize_binary_message(msg_bits),
        "pubkey_bytes": _normalize_pubkey(pubkey_bytes),
        "salt": shift_data["salt"],
        "k": k,
        "sigma": s,
        "coeffs": shift_data["coeffs"],
        "x": shift_data["x"],
        "h": h,
        "t": t,
        "sample": sample,
        "sample_minus_t": sample - vector(QQ, t),
        "w": w,
        "s_vec": s_vec,
        "w_minus_h_even": all((w[i] - h[i]) % 2 == 0 for i in range(len(h))),
    }

In [6]:
message = "10100100101101100001"
pubkey = bytes.fromhex("aabbccddeeff00112233445566778899")
fixed_salt = bytes.fromhex("00112233445566778899aabbccddeeff")
k = 64
s = 2.2
coeff_bytes = 4

signature = hawk_style_e8_sign(message, pubkey, s=s, salt=fixed_salt, k=k, coeff_bytes=coeff_bytes)
print("k:", signature["k"])
print("ambient dimension:", len(signature["t"]))
print("salt (hex):", signature["salt"].hex())
print("quotient representative t:", signature["t"])
print("sampler output:", signature["sample"])
print("basis coordinates w:", signature["w"])
print("binary challenge h:", signature["h"])
print("signature vector s:", signature["s_vec"])
print("w-h even componentwise:", signature["w_minus_h_even"])
print("sample lies in t + 2E8^k:", in_direct_sum_rep_plus_2E8(signature["sample"], signature["t"]))


signature_2 = hawk_style_e8_sign(message, pubkey, s=s, salt=fixed_salt, k=k, coeff_bytes=4)
print("\nDeterministic t for fixed transcript inputs:", signature["t"] == signature_2["t"])
print("Independent sampling with same t can differ:", signature["sample"] != signature_2["sample"])

k: 64
ambient dimension: 512
salt (hex): 00112233445566778899aabbccddeeff
quotient representative t: (1, 1, 1, 3, 0, 2, 2, 2, 0, 0, 0, 2, 1, 1, 3, 1, 1, 1, 0, 4, 1, 5, 2, 4, 1, 0, 0, 1, 0, 3, 3, 2, 1, 1, 1, 5, 1, 5, 5, 3, 0, 1, 1, 4, 1, 2, 4, 3, 0, 1, 1, 2, 1, 4, 2, 5, 0, 1, 0, 1, 0, 3, 2, 3, 1, 1, 0, 2, 0, 4, 3, 1, 0, 0, 0, 0, 1, 1, 3, 1, 0, 0, 1, 3, 1, 3, 4, 4, 0, 0, 0, 2, 0, 0, 0, 0, 1, 0, 0, 1, 0, 3, 3, 2, 0, 1, 0, 1, 0, 1, 2, 3, 0, 0, 0, 2, 1, 3, 1, 3, 0, 1, 0, 3, 0, 1, 2, 3, 0, 0, 0, 2, 0, 0, 0, 0, 1, 0, 0, 1, 1, 2, 2, 3, 0, 1, 0, 1, 1, 2, 1, 2, 0, 0, 1, 1, 0, 2, 3, 3, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 2, 1, 1, 1, 1, 0, 1, 0, 1, 1, 4, 3, 4, 1, 1, 0, 4, 1, 5, 2, 2, 1, 1, 0, 4, 1, 5, 4, 2, 0, 0, 1, 3, 1, 1, 2, 2, 1, 0, 1, 4, 1, 4, 5, 4, 1, 1, 1, 5, 1, 5, 5, 5, 0, 1, 0, 3, 0, 3, 0, 3, 1, 0, 1, 4, 1, 4, 5, 2, 1, 0, 1, 2, 1, 2, 5, 4, 0, 1, 1, 4, 0, 1, 3, 4, 0, 0, 1, 1, 1, 3, 2, 4, 1, 0, 0, 3, 1, 4, 4, 3, 0, 0, 1, 1, 1, 3, 2, 2, 1, 0, 1, 2, 0, 1, 4, 1, 1, 0, 0, 3, 0, 1, 3, 0, 1, 1, 0, 2